# Tutorial & Tugas UTS: EDA dan Pra-pemrosesan dengan Dask & GCS di Google Colab

## Tujuan Pembelajaran

Dalam tutorial ini, Anda akan belajar cara melakukan analisis data eksplorasi (EDA) dan tugas UTS pra-pemrosesan data menggunakan Dask. Kita akan memproses kumpulan data besar yang disimpan di Google Cloud Storage (GCS) langsung dari Google Colab.

## Bagian 1: Persiapan Environment

### 1. Install Library
Jalankan sel kode berikut untuk menginstal pustaka yang diperlukan (dask).

In [1]:
!pip install gcsfs "dask[complete]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.8 MB/s eta 0:00:00


### 2. Mulai Klaster Dask
Inisialisasi LocalCluster dan Client. Dask akan menggunakan core yang tersedia di mesin virtual Colab.

In [ ]:
from dask.distributed import Client, LocalCluster
import dask.dataframe as dd

# Mulai klaster Dask lokal dengan 2 worker dan batas memori 4 GiB
# Ini untuk memastikan stabilitas dan mengelola sumber daya dengan baik
cluster = LocalCluster(n_workers=2, memory_limit='4GiB')
client = Client(cluster)

print(f"Dasbor Dask tersedia di: {client.dashboard_link}")

### 3. Muat Data dari GCS
Akses data penerbangan publik dari GCS. Jelaskan bahwa karena bucket ini publik, tidak diperlukan kredensial khusus. Dask secara otomatis menggunakan gcsfs untuk mengakses data.

In [ ]:
# Tentukan path GCS ke data
gcs_path = "gcs://quansight-datasets/airline-ontime-performance/csv/*ber_2020.csv"


# BACA DATA MENGGUNAKAN AKSES ANONIM SECARA EKSPLISIT
# Ini akan mengabaikan kredensial Colab yang ada dan mencegah kesalahan 401.
df = dd.read_csv(gcs_path, assume_missing=True, storage_options={'token': 'anon'})
# df = dd.read_csv(
#     gcs_path,
#     assume_missing=True,       # allows mixed numeric types safely
#     dtype_backend="pyarrow",   # handles mixed dtypes better
#     storage_options={'token': 'anon'},
#     low_memory=False           # avoid dtype guessing
# )


## Bagian 2: Analisis Data Eksplorasi (EDA) dengan Dask

### 1. Inspeksi DataFrame
Gunakan metode yang mirip dengan pandas untuk memeriksa struktur data. Pada saat yang diperlukan, gunakan .compute() untuk menjalankan penghitungan.

In [ ]:
df = dd.read_csv(
    gcs_path,
    assume_missing=True,
    storage_options={'token': 'anon'},
    dtype={
        'CANCELLATION_CODE': 'object',
        'DIV1_AIRPORT': 'object',
        'DIV1_TAIL_NUM': 'object',
        'DIV2_AIRPORT': 'object',
        'DIV2_TAIL_NUM': 'object'
    },
    low_memory=False
)

In [ ]:
# Tampilkan beberapa baris pertama dari Dask DataFrame
print(df.head())

# Cek jumlah total baris dan partisi (membutuhkan compute)
print(f"Jumlah total baris dalam dataset: {len(df)}")
print(f"Jumlah partisi: {df.npartitions}")


### 2. Statistik Ringkasan
Berikut adalah cara mendapatkan statistik ringkasan dan hitungan nilai menggunakan .compute().

In [ ]:
# Hitung statistik deskriptif untuk kolom numerik
print(df.describe().compute())

# Hitung jumlah nilai unik untuk kolom 'ORIGIN'
print(df['ORIGIN'].value_counts().compute().head())


In [ ]:
df['YEAR'].head()

### 3. Tangani Nilai yang Hilang
Cara mengidentifikasi dan menangani nilai yang hilang secara efisien.

In [ ]:
# Hitung persentase nilai kosong per kolom
null_counts = df.isnull().sum().compute() / len(df)
print(null_counts[null_counts > 0])

# Ganti nilai kosong di kolom 'FL_NUM' dengan nilai rata-rata
df['DEP_DELAY'] = df['DEP_DELAY'].fillna(df['DEP_DELAY'].mean())


## Bagian 3: Pra-pemrosesan Data dengan Dask

### 1. Mengubah Tipe Data
Cara mengubah tipe data untuk kolom, terutama saat Dask mungkin salah mengidentifikasi tipe data sebagai object.

In [ ]:
# Ubah kolom 'YEAR' menjadi tipe integer
df['YEAR'] = df['YEAR'].astype(int)

# Ubah kolom bertipe object menjadi tipe category untuk efisiensi
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].astype('category')


### 2. Rekayasa Fitur
Membuat fitur baru dari kolom yang sudah ada.

In [ ]:
# Buat fitur baru untuk durasi penerbangan
df['FLIGHT_DURATION'] = df['ARR_TIME'] - df['DEP_TIME']

### 3. Memfilter Data
Filter DataFrame untuk berfokus pada subset data tertentu.

In [ ]:
# Filter penerbangan di tahun 2020 yang mengalami keterlambatan
delayed_flights_2020 = df[(df['YEAR'] == 2020) & (df['ARR_DELAY'] > 0)]

# Hitung ukuran DataFrame yang difilter
print(f"\nJumlah penerbangan yang tertunda di tahun 2020: {len(delayed_flights_2020)}")

## Bagian 4: Visualisasi dengan Dask dan Matplotlib

### 1. Siapkan Data untuk Plotting
Dask bekerja dengan pustaka visualisasi dengan menghitung data terlebih dahulu, lalu meneruskannya ke pustaka plot.

In [ ]:
df.OP_UNIQUE_CARRIER.head()

In [ ]:
import matplotlib.pyplot as plt

# Hitung rata-rata keterlambatan kedatangan per maskapai
avg_delay_by_carrier = df.groupby('OP_UNIQUE_CARRIER')['ARR_DELAY'].mean().compute()

# Plot hasilnya menggunakan pandas dan matplotlib
plt.figure(figsize=(12, 6))
avg_delay_by_carrier.plot(kind='bar')
plt.title('Rata-rata Keterlambatan Kedatangan per Maskapai')
plt.xlabel('Maskapai')
plt.ylabel('Rata-rata Keterlambatan Kedatangan (menit)')
plt.show()


## Bagian 5: Tugas UTS
Sekarang giliran Anda! Jawab pertanyaan-pertanyaan di bawah ini menggunakan Dask di Google Colab.

### Pertanyaan Konseptual
Jelaskan dengan kata-kata Anda sendiri mengapa Dask diperlukan untuk dataset ini dan masalah apa yang dipecahkannya dibandingkan dengan menggunakan alat single-core seperti pandas.

Jawaban:
Dask diperlukan karena dataset penerbangan ini ukurannya sangat besar sehingga tidak muat untuk diproses sekaligus di memori RAM jika menggunakan pandas.
Pandas bekerja secara single-core dan memuat seluruh data ke memori, sehingga akan lambat dan mudah membuat notebook kehabisan RAM.

Dask memecahkan masalah ini dengan cara:
1. Membagi dataset besar menjadi banyak partisi kecil sehingga bisa diproses bertahap (lazy evaluation).
2. Dapat menjalankan komputasi secara paralel menggunakan multi-core (lebih cepat daripada pandas).
3. Tidak perlu memuat seluruh dataset ke memori; Dask hanya membaca bagian yang diperlukan.
4. Dapat membaca data langsung dari cloud seperti GCS tanpa harus mengunduh file besar terlebih dahulu.

Karena itu, Dask jauh lebih efisien untuk EDA dan preprocessing dataset besar seperti data penerbangan ini.


#### Tuliskan Nama dan NIM di sini

- NAMA: #Muhammad Wildan Baihaqi
- NIM: #202210370311151

### Pertanyaan Konseptual

1. Jelaskan dengan kata-kata Anda sendiri mengapa Dask diperlukan untuk dataset ini dan masalah apa yang dipecahkannya dibandingkan dengan menggunakan package single-core seperti pandas.

##### Jawaban:
## Jawab Pertanyaan-2 pada baris berikutnya (baris ini jangan dihapus)
avg_dep_delay = df.groupby('OP_UNIQUE_CARRIER')['DEP_DELAY'].mean().compute()
avg_dep_delay.sort_values(ascending=False).head()



### Pertanyaan Pemrograman

2. Hitung rata-rata keterlambatan keberangkatan (DEP_DELAY) untuk setiap maskapai (OP_UNIQUE_CARRIER) dalam dataset. Tampilkan 5 maskapai teratas dengan rata-rata keterlambatan tertinggi.

In [ ]:
# Jawab Pertanyaan-2 pada baris berikutnya (baris ini jangan dihapus)
avg_dep_delay = df.groupby('OP_UNIQUE_CARRIER')['DEP_DELAY'].mean().compute()
avg_dep_delay.sort_values(ascending=False).head()


3. Menggunakan kolom FLIGHT_DURATION yang telah dibuat, filter DataFrame untuk menemukan semua penerbangan yang durasinya lebih dari 5 jam. Kemudian, hitung dan tampilkan jumlah total penerbangan tersebut.

In [ ]:
# Jawab Pertanyaan-3 pada baris berikutnya (baris ini jangan dihapus)
long_flights = df[df['FLIGHT_DURATION'] > 300]
len(long_flights)


4. Buat plot yang menampilkan jumlah total penerbangan yang tertunda (ARR_DELAY > 0) per tahun. Pastikan untuk melabeli plot dengan jelas.

In [ ]:
# Jawab Pertanyaan-4 pada baris berikutnya (baris ini jangan dihapus)
delayed_per_year = df[df['ARR_DELAY'] > 0].groupby('YEAR').size().compute()
delayed_per_year.plot(kind='bar', figsize=(10,5))
plt.title("Jumlah Penerbangan Tertunda per Tahun")
plt.xlabel("Tahun")
plt.ylabel("Jumlah Penerbangan Tertunda")
plt.show()


5. Pertanyaan Bonus: Buat fitur baru bernama FLIGHT_SPEED (dalam mil per menit) dan gunakan untuk menemukan penerbangan dengan kecepatan tertinggi yang tercatat.

In [ ]:
# Jawab Pertanyaan-5 pada baris berikutnya (baris ini jangan dihapus)
df['FLIGHT_SPEED'] = df['DISTANCE'] / df['FLIGHT_DURATION']
df['FLIGHT_SPEED'].max().compute()


### Bagian 6: Pembersihan
Setelah selesai, tutup klaster Dask.

In [ ]:
client.close()
cluster.close()